In [70]:
import pandas as pd
import numpy as np

from pycaret.clustering import (
    setup as cl_setup,
    create_model as cl_create_model,
    save_model as cl_save_model,
)

from pycaret.regression import (
    setup as reg_setup,
    save_model as reg_save_model,
    compare_models as reg_compare_models
)

In [71]:
# wczytanie danych z pliku csv

df = pd.read_csv("insurance_data.csv", delimiter=",")
df.columns.tolist()

['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']

In [72]:
# czyszczenie nazw kolumn
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df.columns.tolist()

['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']

In [73]:
# sprawdzanie czy są brakujące wartości i ich usuwanie z DataFrame, kolumna po kolumnie

empty_values = df.isna().sum()

for key, value in empty_values.items():
    # print(f"{key}: {value}")
    if value > 0:
        df.dropna(subset=[key], inplace=True)

In [74]:
# cluster_data = df.drop(columns=['charges'])  # replace 'column_to_drop' with the actual column name you want to drop
# cluster_data.columns.tolist()

---
# **Klasteryzacja danych**

In [75]:
# Ustawienia klasteryzacji danych przy użyciu PyCaret

clustering_setup = cl_setup(
    data=df,
    session_id=123,
    normalize=True, # normalizacja danych przed klasteryzacją
    verbose=True, # wyświetlanie szczegółowych informacji podczas ustawiania klasteryzacji
    ignore_features=['charges'], # ignorowanie kolumny 'charges' podczas klasteryzacji
)

,Description,Value
0,Session id,123
1,Original data shape,"(1338, 7)"
2,Transformed data shape,"(1338, 9)"
3,Ignore features,1
4,Numeric features,3
5,Categorical features,3
6,Preprocess,True
7,Imputation type,simple
8,Numeric imputation,mean
9,Categorical imputation,mode


In [76]:
# stworzenie modelu k-means z 3 klastrami

model = cl_create_model('kmeans', num_clusters=3)


,Silhouette,Calinski-Harabasz,Davies-Bouldin,Homogeneity,Rand Index,Completeness
0,0.2496,295.5350,1.7141,0,0,0


In [ ]:
# zapis pipeline
cl_save_model(model, 'pipeline_clustering')
print()

Transformation Pipeline and Model Successfully Saved



---
# **Regresja**

In [78]:
# przygotowanie danych do analizy regresji
df_reg = df.copy()

# sprawdzenie typów wybranych kolumn i ewentualna konwersja do typów numerycznych
for column, dtype in dict(df_reg[['age', 'bmi', 'children', 'charges']].dtypes).items():
    if not np.issubdtype(dtype, np.number):
        df_reg[column] = pd.to_numeric(df_reg[column], errors='coerce')

    print(f"kolumna [{column}] jest teraz typu [{df_reg[column].dtype}]")

kolumna [age] jest teraz typu [int64]
kolumna [bmi] jest teraz typu [float64]
kolumna [children] jest teraz typu [int64]
kolumna [charges] jest teraz typu [float64]


In [79]:
regression_setup = reg_setup(
    data=df_reg,
    target='charges',
    session_id=123,
    normalize=True, # -> dane wejściowe są skalowane
    verbose=False # -> tryb cichy, minimalny output.
)

In [ ]:
# porównanie modeli regresyjnych i wybór najlepszego

best_reg_model = reg_compare_models()
best_reg_model

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
gbr,Gradient Boosting Regressor,2704.9411,23567157.5677,4834.8275,0.8319,0.4448,0.3140,0.0430
rf,Random Forest Regressor,2772.7847,25379240.4648,5024.5579,0.8174,0.4685,0.3304,0.0770
lightgbm,Light Gradient Boosting Machine,2989.9577,25391993.0970,5029.6985,0.8157,0.5442,0.3750,0.0930
et,Extra Trees Regressor,2833.3624,28427844.2412,5305.6516,0.7991,0.4877,0.3363,0.0670
ada,AdaBoost Regressor,4198.3012,28689735.5532,5348.5728,0.7955,0.6291,0.7156,0.0240
knn,K Neighbors Regressor,3558.0286,33757213.4000,5798.2844,0.7605,0.4940,0.3628,0.0310
lar,Least Angle Regression,4303.5559,38388058.4578,6176.5920,0.7306,0.5949,0.4433,0.0330
llar,Lasso Least Angle Regression,4303.4005,38387422.4107,6176.5378,0.7306,0.5941,0.4433,0.0190
br,Bayesian Ridge,4308.1432,38390146.8642,6176.7504,0.7306,0.5869,0.4443,0.0210
ridge,Ridge Regression,4305.1268,38388171.4400,6176.5963,0.7306,0.5911,0.4437,0.0210


c:\programy\miniconda3\envs\od_zera_do_ai\Lib\site-packages\pycaret\internal\pycaret_experiment\supervised_experiment.py:339: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(highlight_cols, subset=["TT (Sec)"])


GradientBoostingRegressor(random_state=123)

### Najlepszym modelem w naszym przypadku okazał się **Gradient Boosting Regressor**

In [ ]:
# zapisanie modelu do pliku pipeline_regression
reg_save_model(best_reg_model, 'pipeline_regression')

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(include=['age', 'bmi', 'children'],
                                     transformer=SimpleImputer())),
                 ('categorical_imputer',
                  TransformerWrapper(include=['sex', 'smoker', 'region'],
                                     transformer=SimpleImputer(strategy='most_frequent'))),
                 ('ordinal_encoding',
                  TransformerWrapper(include=['sex', 'smoker'],
                                     transfor...
                                                                         {'col': 'smoker',
                                                                          'data_type': dtype('O'),
                                                                          'mapping': no     0
 yes    1
 NaN   -1
 dtype: int64}]))),
                 ('onehot_encoding',
                  TransformerWrapper(include=['region'],
    